# Power Map Activation Function Visualization

Visualizing the smooth activation function and the battery power map:

$$\sigma_k(u) = \frac{1}{2}\left(1 + \tanh(ku)\right)$$

$$P_t = P_{\max}\left[\sigma_k(u - \epsilon)\sigma_k(0.995 - SOC_t)u + \sigma_k(-u - \epsilon)\sigma_k(SOC_t - 0.005)u\right]$$

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

k = 500
epsilon = 0.0025
soc_min = 0.005
soc_max = 0.995


def sigma_k(x, k=k):
    return 0.5 * (1 + np.tanh(k * x))


def charging_branch_map(u, soc, k=k, epsilon=epsilon):
    return sigma_k(u - epsilon, k) * sigma_k(soc_max - soc, k) * u


def discharging_branch_map(u, soc, k=k, epsilon=epsilon):
    return sigma_k(-u - epsilon, k) * sigma_k(soc - soc_min, k) * u


def normalized_power_map(u, soc, k=k, epsilon=epsilon):
    charging_branch = charging_branch_map(u, soc, k=k, epsilon=epsilon)
    discharging_branch = discharging_branch_map(u, soc, k=k, epsilon=epsilon)
    return charging_branch + discharging_branch


output_dir = Path.cwd()
if output_dir.name != "notebooks" and (output_dir / "notebooks").exists():
    output_dir = output_dir / "notebooks"

activation_path = output_dir / "power_map_activation_gates.png"
boundary_path = output_dir / "power_map_boundary_gates.png"
k_effect_path = output_dir / "power_map_k_epsilon_effect.png"

# Figure 1: charging and discharging activation gates.
x = np.linspace(-0.02, 0.02, 600)
fig1, ax1 = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
ax1.plot(
    x,
    sigma_k(-x - epsilon),
    color="tab:blue",
    linewidth=1.5,
    label=rf"$\sigma_k(-u - \epsilon)$, $\epsilon={epsilon}$",
)
ax1.plot(
    x,
    sigma_k(x - epsilon),
    color="tab:orange",
    linewidth=1.5,
    label=rf"$\sigma_k(u - \epsilon)$, $\epsilon={epsilon}$",
)
ax1.axvline(0, color="0.35", linestyle="--", linewidth=1)
ax1.axvline(epsilon, color="black", linestyle=":", linewidth=1.2)
ax1.axvline(-epsilon, color="black", linestyle=":", linewidth=1.2)
ax1.set_title(r"Charging and discharging activation gates")
ax1.set_xlabel(r"$u$")
ax1.set_ylabel(r"$\sigma_k(\cdot)$")
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, alpha=0.28)
ax1.legend(loc="lower right", fontsize=8, frameon=False)
fig1.savefig(activation_path, dpi=200, bbox_inches="tight")

# Figure 2: boundary branch behavior.
u_empty = np.linspace(-0.02, 0.01, 500)
u_full = np.linspace(-0.01, 0.02, 500)
low_soc = np.linspace(0, 0.02, 350)
high_soc = np.linspace(0.98, 1.0, 350)
U_empty, SOC_low = np.meshgrid(u_empty, low_soc)
U_full, SOC_high = np.meshgrid(u_full, high_soc)
P_empty = discharging_branch_map(U_empty, SOC_low)
P_full = charging_branch_map(U_full, SOC_high)

fig2, axes2 = plt.subplots(2, 1, figsize=(8, 7.4), constrained_layout=True)
heatmap = axes2[0].pcolormesh(
    u_empty, low_soc, P_empty, shading="auto", cmap="coolwarm", vmin=-0.02, vmax=0.02
)
axes2[0].axvline(0, color="0.35", linestyle="--", linewidth=1)
axes2[0].axvline(-epsilon, color="black", linestyle=":", linewidth=1.2)
axes2[0].axhline(soc_min, color="black", linestyle=":", linewidth=1.2)
axes2[0].set_title(r"Near-empty boundary: discharging gate $\sigma_k(-u - \epsilon)$")
axes2[0].set_xlabel(r"control $u$")
axes2[0].set_ylabel(r"$SOC_t$")

axes2[1].pcolormesh(
    u_full, high_soc, P_full, shading="auto", cmap="coolwarm", vmin=-0.02, vmax=0.02
)
axes2[1].axvline(0, color="0.35", linestyle="--", linewidth=1)
axes2[1].axvline(epsilon, color="black", linestyle=":", linewidth=1.2)
axes2[1].axhline(soc_max, color="black", linestyle=":", linewidth=1.2)
axes2[1].set_title(r"Near-full boundary: charging gate $\sigma_k(u - \epsilon)$")
axes2[1].set_xlabel(r"control $u$")
axes2[1].set_ylabel(r"$SOC_t$")
fig2.colorbar(heatmap, ax=axes2, label=r"branch contribution / $P_{\max}$")
fig2.suptitle("SOC boundary branch behavior", fontsize=13)
fig2.savefig(boundary_path, dpi=200, bbox_inches="tight")

# Figure 3: effect of k and epsilon on the shifted activation gate.
x_zoom = np.linspace(-0.02, 0.02, 800)
fig3, ax3 = plt.subplots(figsize=(8, 4.1), constrained_layout=True)
for k_value in [50, 150, 500, 1500]:
    ax3.plot(
        x_zoom,
        sigma_k(x_zoom - epsilon, k=k_value),
        linewidth=2,
        label=rf"$k={k_value}$, $\epsilon={epsilon}$",
    )
for epsilon_value in [0.0, 0.005, 0.01]:
    ax3.plot(
        x_zoom,
        sigma_k(x_zoom - epsilon_value, k=k),
        linestyle="--",
        linewidth=2,
        label=rf"$k={k}$, $\epsilon={epsilon_value}$",
    )
ax3.axvline(0, color="0.35", linestyle="--", linewidth=1)
ax3.axvline(epsilon, color="black", linestyle=":", linewidth=1)
ax3.set_title(r"Effect of $k$ and $\epsilon$ on $\sigma_k(u - \epsilon)$")
ax3.set_xlabel(r"$u$")
ax3.set_ylabel(r"$\sigma_k(u - \epsilon)$")
ax3.set_ylim(-0.05, 1.05)
ax3.grid(True, alpha=0.28)
ax3.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=8, frameon=False, ncol=2)
fig3.savefig(k_effect_path, dpi=200, bbox_inches="tight")

plt.show()

activation_path.resolve(), boundary_path.resolve(), k_effect_path.resolve()


ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']